## 📓 Notebook 03: Encoding Categoricals

**Skills practiced:** One-hot encoding, label encoding for ordered categories, target encoding computed from train only.

**Goal:** Encode each categorical column with the method that fits its nature, and explain why one-hot, label, or target encoding was right for each.

In [20]:
import pandas as pd
import numpy as np


from sklearn.preprocessing import OneHotEncoder, LabelEncoder

from sklearn.metrics import root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor

In [21]:
x_train=pd.read_csv('../data/processed/stage_1/x_train.csv')
x_test=pd.read_csv('../data/processed/stage_1/x_test.csv')
y_train=pd.read_csv('../data/processed/y_train.csv')
y_test=pd.read_csv('../data/processed/y_test.csv')

**weather_code**: Category of weather conditions.

| Code | Meaning |
|------|---------|
| 1 | Clear; mostly clear but may have some values with haze/fog/patches of fog/fog in vicinity |
| 2 | Scattered clouds/few clouds |
| 3 | Broken clouds |
| 4 | Cloudy |
| 7 | Rain/light rain shower/light rain |
| 10 | Rain with thunderstorm |
| 26 | Snowfall |
| 94 | Freezing fog |

**season**: Categorical field indicating meteorological seasons.

| Code | Meaning |
|------|---------|
| 0 | Spring |
| 1 | Summer |
| 2 | Fall |
| 3 | Winter |

In [22]:
x_train['weather_code'].nunique()

7

In [23]:
x_train['season'].nunique()

4

In [24]:
labels = {
    'weather_code': {
        1: 'Clear',
        2: 'Scattered clouds',
        3: 'Broken clouds',
        4: 'Cloudy',
        7: 'Light rain',
        10: 'Rain with thunderstorm',
        26: 'Snowfall',
        94: 'Freezing fog'
    },
    'season': {
        0: 'Spring',
        1: 'Summer',
        2: 'Fall',
        3: 'Winter'
    }
}

x_train['season']=x_train['season'].map(labels['season'])
x_train['weather']=x_train['weather_code'].map(labels['weather_code'])
x_train=x_train.drop(columns=['weather_code'])

x_test['season']=x_test['season'].map(labels['season'])
x_test['weather']=x_test['weather_code'].map(labels['weather_code'])
x_test=x_test.drop(columns=['weather_code'])

In [25]:
ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")

cols = ["weather", "season"]

train_cat = pd.DataFrame(
    ohe.fit_transform(x_train[cols]),
    columns=ohe.get_feature_names_out(cols),
    index=x_train.index)

test_cat = pd.DataFrame(
    ohe.transform(x_test[cols]),
    columns=ohe.get_feature_names_out(cols),
    index=x_test.index)

x_train = pd.concat([x_train.drop(columns=cols), train_cat], axis=1)
x_test = pd.concat([x_test.drop(columns=cols), test_cat], axis=1)

In [26]:
x_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 13931 entries, 0 to 13930
Data columns (total 18 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   t1                              13931 non-null  float64
 1   t2                              13931 non-null  float64
 2   hum                             13931 non-null  float64
 3   wind_speed                      13931 non-null  float64
 4   is_holiday                      13931 non-null  float64
 5   is_weekend                      13931 non-null  float64
 6   hour                            13931 non-null  int64  
 7   weather_Broken clouds           13931 non-null  float64
 8   weather_Clear                   13931 non-null  float64
 9   weather_Cloudy                  13931 non-null  float64
 10  weather_Light rain              13931 non-null  float64
 11  weather_Rain with thunderstorm  13931 non-null  float64
 12  weather_Scattered clouds        13931 non-n

In [27]:
y_train_cls, edges = pd.qcut(y_train['cnt'], q=3, labels=['low','medium','high'], retbins=True)

edges[0], edges[-1] = -np.inf, np.inf
y_test_cls = pd.cut(y_test['cnt'], bins=edges, labels=['low','medium','high'])

In [28]:
mapping = {'low': 0, 'medium': 1, 'high': 2}
y_train_enc = y_train_cls.map(mapping).astype(int)
y_test_enc = y_test_cls.map(mapping).astype(int)

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train_cls)
y_test_enc = le.transform(y_test_cls)
print(le.classes_)

y_train = pd.concat([y_train[['cnt']], pd.get_dummies(y_train_cls).astype(int)], axis=1)
y_test = pd.concat([y_test[['cnt']], pd.get_dummies(y_test_cls).astype(int)], axis=1)

['high' 'low' 'medium']


In [29]:
print(y_train.shape)
print(y_test.shape)

(13931, 4)
(3483, 4)


In [30]:
y=y_train['cnt']

m=RandomForestRegressor(n_estimators=250, random_state=42)
m.fit(x_train,y)
preds=m.predict(x_test)

RMSE=root_mean_squared_error(y_test['cnt'], preds)
print(round(RMSE,2))

250.2


In [31]:
x_train.to_csv('../data/processed/stage_2/x_train_enc.csv', index=False)
x_test.to_csv('../data/processed/stage_2/x_test_enc.csv', index=False)